# Extended Depth of Field

Combine photographs focused at different distances into one composite. Inputs must show the same scene, be aligned, and have the same dimensions.

The focus measure is the absolute Laplacian after a 5×5 Gaussian blur. Each output pixel comes from the image with the strongest response; ties choose the first input. This simple method does not align images or smooth selection boundaries.

## 1. Setup

Install `requirements.txt` in a virtual environment. Open this notebook from the repository root or the `notebooks/` directory and run all cells in order. Paths below are resolved from the repository root. OpenCV uses BGR channel order throughout.

In [ ]:
from pathlib import Path

import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np

plt.style.use("ggplot")

# Support kernels started from the repository root or notebooks directory.
working_directory = Path.cwd().resolve()
project_root = (
    working_directory.parent
    if working_directory.name == "notebooks"
    else working_directory
)
if not (project_root / "data" / "examples").is_dir():
    raise FileNotFoundError(
        "Start the notebook from the repository root or notebooks directory."
    )
input_directory = project_root / "data" / "examples"
output_directory = project_root / "outputs"


## 2. Load source images

Input order determines the winner when focus responses are equal.

In [ ]:
image_paths = [input_directory / f"img_{index:02d}.jpg" for index in range(1, 5)]
images = []
for path in image_paths:
    image = cv.imread(str(path))
    if image is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    images.append(image)

## 3. Measure focus

`cv.CV_32F` explicitly names the original Laplacian depth (`5`). NumPy selects all pixels at once while retaining the original tie behavior.

In [ ]:
def compute_focus_maps(images):
    """Return signed float32 Laplacian maps for equally sized BGR images."""
    if not images:
        raise ValueError("Provide at least one image.")
    shape = images[0].shape
    if any(image.shape != shape for image in images):
        raise ValueError("All images must have the same shape.")
    focus_maps = []
    for image in images:
        gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
        blurred = cv.GaussianBlur(gray, (5, 5), 0)
        focus_maps.append(cv.Laplacian(blurred, cv.CV_32F))
    return np.asarray(focus_maps)


def select_focused_pixels(images, focus_maps):
    """Copy the strongest-response pixel, choosing the first image on ties."""
    best_indices = np.abs(focus_maps).argmax(axis=0)
    # Gather one BGR pixel per location without Python pixel loops.
    rows, columns = np.indices(best_indices.shape)
    return np.asarray(images)[best_indices, rows, columns]


def focus_stack(images):
    """Combine registered BGR images using the original focus measure."""
    return select_focused_pixels(images, compute_focus_maps(images))


In [ ]:
focus_maps = compute_focus_maps(images)
fig, axes = plt.subplots(2, 2, figsize=(15, 15), sharex=True, sharey=True)
for axis, path, focus_map in zip(axes.flat, image_paths, focus_maps):
    axis.imshow(focus_map, cmap="gray")
    axis.set_title(path.name)
    axis.axis("off")
plt.show()

## 4. Compose and save

The composite is saved as `outputs/infocus_img.jpg` relative to the repository root. The output directory is created automatically. The plot uses BGR channel order; use `cv.cvtColor(focused_image, cv.COLOR_BGR2RGB)` for a true-color Matplotlib preview.

In [ ]:
focused_image = select_focused_pixels(images, focus_maps)
output_directory.mkdir(parents=True, exist_ok=True)
output_path = output_directory / "infocus_img.jpg"
if not cv.imwrite(str(output_path), focused_image):
    raise OSError(f"Could not write image: {output_path}")

plt.figure(figsize=(10, 10))
plt.imshow(focused_image)
plt.axis("off")
plt.show()

## 5. Reproduce the benchmark

Run `python benchmarks/benchmark.py --repeats 5` from the repository root. It compares the preserved baseline with these notebook functions, checks exact pixel and JPEG equality, and writes machine-readable results to `benchmarks/results.json`. The acceptance threshold and recorded measurements are in `benchmarks/results.json`.